# Pipeline de pre-processamento configuravel por run

Este notebook demonstra como montar o pipeline de pre-processamento
recomendado pelo livro-texto (Hyvarinen, Karhunen & Oja, *Independent
Component Analysis*, Capitulo 13, "Practical Considerations", p.263-272)
de forma que **cada run escolha so os passos que precisa**: os passos
"normais" (centralizacao + branqueamento) sempre entram; PCA e filtragem
temporal sao opcionais e podem ser adicionados ou removidos livremente.

Resumo do que o livro propoe (ver `context/ICA_BACKGROUND.md` e a
investigacao anterior desta conversa):

- **Centralizacao + branqueamento**: sempre (Secao 2 de `ICA_BACKGROUND.md`).
- **PCA** (Secao 13.2, p.267-269): reduz a dimensao quando ha mais misturas
  do que fontes, e/ou reduz ruido e evita *overlearning* descartando
  direcoes de baixa variancia.
- **Filtragem temporal** (Secao 13.1, p.263-267): so faz sentido para
  sinais com estrutura de tempo real (audio). Passa-baixa reduz ruido;
  passa-alta aproxima o processo de inovacao (Teorema 13.1), aumentando
  independencia e nao-gaussianidade.

A metrica de nao-gaussianidade (curtose de Fisher) usada em todo o
notebook e `ica.metrics.non_gaussianity.NonGaussianityScore`, ja
implementada e testada em `src/ica/metrics/non_gaussianity.py` -- aqui ela
e comparada com a curtose das misturas *antes* da ICA (via
`scipy.stats.kurtosis` diretamente), reproduzindo o teste de
nao-gaussianidade discutido na investigacao do livro-texto (no maximo uma
fonte pode ser gaussiana; Secao 7.5).

In [1]:
import numpy as np
from pathlib import Path
from scipy.stats import kurtosis

from ica.algorithms.fastica_ml import FastICAML
from ica.data.audio_template import AudioTemplate
from ica.data.base import DataTemplate
from ica.data.distribution_template import DistributionTemplate
from ica.data.image_template import ImageTemplate
from ica.metrics.convergence_iterations import ConvergenceIterations
from ica.metrics.execution_time import ExecutionTime
from ica.metrics.log_likelihood import LogLikelihood
from ica.metrics.non_gaussianity import NonGaussianityScore
from ica.model import ICAModel
from ica.nonlinearities.adaptive import AdaptiveScore
from ica.preprocessing.centering import Centering
from ica.preprocessing.pca import PCA
from ica.preprocessing.pipeline import Pipeline
from ica.preprocessing.temporal_filtering import TemporalFiltering
from ica.preprocessing.whitening import Whitening
from ica.visualization.audio_visualizer import AudioVisualizer
from ica.visualization.histogram_visualizer import HistogramVisualizer
from ica.visualization.image_visualizer import ImageVisualizer
from ica.visualization.log_likelihood_visualizer import LogLikelihoodVisualizer
from ica.visualization.mixing_diagram_3d_visualizer import MixingDiagram3DVisualizer
from ica.visualization.mixing_diagram_visualizer import MixingDiagramVisualizer

DATA_ROOT = Path("..") / "data"
OUTPUT_ROOT = Path("..") / "output" / "notebook_demo"


## Pipeline configuravel

`build_pipeline` monta a sequencia de passos para uma run especifica.
`temporal_filtering` e `pca` sao `None` por padrao -- ou seja, **omitir um
argumento e a forma de remover aquele passo do pipeline**, seguindo apenas
os passos normais (centralizacao + branqueamento). Para incluir um passo,
basta passar a instancia ja configurada.

In [2]:
def build_pipeline(
    temporal_filtering: TemporalFiltering | None = None,
    pca: PCA | None = None,
) -> Pipeline:
    """Monta o pipeline de pre-processamento para uma run especifica.

    Os passos normais (Centering + Whitening) sao sempre incluidos. A
    ordem segue a Secao 13 do livro-texto: filtrar (se aplicavel) ->
    reduzir dimensao (se aplicavel) -> branquear -- TemporalFiltering
    entra antes de PCA/Whitening para que a covariancia usada na reducao
    de dimensao e no branqueamento seja estimada a partir do sinal ja
    filtrado (mais limpo), sem que isso afete a reconstrucao final das
    fontes (TemporalFiltering e estimation_only, ver
    ica.preprocessing.base.PreprocessingStep).

    Parameters
    ----------
    temporal_filtering : TemporalFiltering or None
        Passe uma instancia configurada (ex.: ``TemporalFiltering(kind="high")``)
        para runs com estrutura temporal real (audio) que se beneficiem de
        filtragem; ``None`` (default) para pular esse passo -- o caso de
        imagens e distribuicoes, cujas "amostras" nao tem ordem temporal
        significativa (Secao 13.1.1).
    pca : PCA or None
        Passe uma instancia configurada (ex.: ``PCA(n_components=3)``)
        para runs com mais misturas do que fontes, ou quando reducao de
        ruido for desejada; ``None`` (default) para pular esse passo --
        o caso de todas as runs deste trabalho, onde o numero de misturas
        ja e igual ao numero de fontes por construcao (ver
        TASK_DESCRIPTION.md).

    Returns
    -------
    Pipeline
        Pipeline pronto para ``ICAModel``.
    """
    steps = [Centering()]
    if temporal_filtering is not None:
        steps.append(temporal_filtering)
    if pca is not None:
        steps.append(pca)
    steps.append(Whitening())
    return Pipeline(steps)


def fit_and_report(
    data: DataTemplate,
    pipeline: Pipeline,
    output_dir: Path,
    visualizers=(),
    max_iterations: int = 500,
):
    """Ajusta um ICAModel (FastICA-ML + chaveamento adaptativo) e reporta metricas-chave.

    Reusa ``NonGaussianityScore`` (ja implementada em
    ``ica.metrics.non_gaussianity``) para a curtose das fontes recuperadas,
    e compara com a curtose das misturas originais (``scipy.stats.kurtosis``
    direto sobre ``model.mixtures_``) -- o teste de nao-gaussianidade "antes
    vs. depois" discutido na investigacao do livro-texto (Secao 7.5: no
    maximo uma fonte pode ser gaussiana).
    """
    model = ICAModel(
        data=data,
        pipeline=pipeline,
        algorithm=FastICAML(nonlinearity=AdaptiveScore(), max_iterations=max_iterations),
    )
    model.fit()

    metrics = model.evaluate(
        [ConvergenceIterations(), ExecutionTime(), NonGaussianityScore(), LogLikelihood()]
    )
    mixture_kurtosis = kurtosis(model.mixtures_, axis=1, fisher=True)

    print(f"pipeline: {[type(step).__name__ for step in pipeline.steps]}")
    print(
        f"convergiu: {model.converged_}  |  {model.n_iterations_} iteracoes  |  "
        f"{metrics['execution_time_seconds']:.3f}s"
    )
    print(f"curtose das misturas   (antes):  {np.round(mixture_kurtosis, 3)}")
    print(f"curtose das fontes     (depois): {np.round(metrics['non_gaussianity_kurtosis'], 3)}")
    print(f"log-verossimilhanca final:       {metrics['log_likelihood']:.3f}")

    output_dir.mkdir(parents=True, exist_ok=True)
    for visualizer in visualizers:
        visualizer.plot(model, output_dir)

    return model, metrics


## 1. Quando PCA e util: demonstracao com dados sinteticos

Nas tres amostras deste trabalho o numero de misturas ja e igual ao
numero de fontes por construcao (ver `TASK_DESCRIPTION.md`), entao PCA
**nao e necessario** para nenhuma run real -- por isso ele fica de fora do
pipeline por padrao (`pca=None`). Para deixar claro quando ele seria
usado (Secao 13.2.1: "making the mixing matrix square"), este exemplo
sintetico mistura 2 fontes em 5 "sensores" (m > n) e usa
`PCA(n_components=2)` para reduzir de volta a dimensao das fontes antes
do branqueamento -- exatamente o teste de integracao em
`tests/integration/test_pca_dimension_reduction.py`.

In [3]:
rng = np.random.default_rng(0)


class _SyntheticArrayTemplate(DataTemplate):
    """DataTemplate minimo que envolve uma matriz ja em memoria, para demonstracao."""

    def __init__(self, X: np.ndarray) -> None:
        super().__init__(run="synthetic", data_root=Path("."))
        self._X = X

    def load(self) -> np.ndarray:
        return self._X

    @property
    def n_mixtures(self) -> int:
        return self._X.shape[0]

    @classmethod
    def discover_runs(cls, data_root: Path) -> list[str]:
        return ["synthetic"]


n_samples = 5000
laplace = rng.laplace(size=n_samples)
uniform = rng.uniform(-1, 1, size=n_samples)
S_demo = np.vstack(
    [(laplace - laplace.mean()) / laplace.std(), (uniform - uniform.mean()) / uniform.std()]
)
A_demo, _ = np.linalg.qr(rng.normal(size=(5, 2)))  # 5 "sensores" captando 2 fontes
X_demo = A_demo @ S_demo

demo_model, demo_metrics = fit_and_report(
    _SyntheticArrayTemplate(X_demo),
    build_pipeline(pca=PCA(n_components=2)),
    output_dir=OUTPUT_ROOT / "pca_demo",
)
print("\nexplained_variance_ratio_ (PCA):", demo_model.pipeline.get_step(PCA).explained_variance_ratio_)
print("shape das fontes recuperadas:", demo_model.sources_.shape, "(reduzido de 5 para 2 misturas)")


pipeline: ['Centering', 'PCA', 'Whitening']
convergiu: True  |  118 iteracoes  |  0.111s
curtose das misturas   (antes):  [-0.41  -1.185  2.962 -0.894  2.905]
curtose das fontes     (depois): [ 2.988 -1.201]
log-verossimilhanca final:       -1.270

explained_variance_ratio_ (PCA): [0.50889358 0.49110642]
shape das fontes recuperadas: (2, 5000) (reduzido de 5 para 2 misturas)


## 2. Amostra de distribuicoes

Sem estrutura temporal (cada linha e uma amostra i.i.d.) e sem excesso de
misturas -- `build_pipeline()` sem argumentos, ou seja, so os passos
normais.

In [4]:
dist_root = DATA_ROOT / "dist"
print("runs disponiveis:", DistributionTemplate.discover_runs(dist_root))

dist_data = DistributionTemplate(run="run1", data_root=dist_root, sample_size=1000)
dist_model, dist_metrics = fit_and_report(
    dist_data,
    build_pipeline(),
    output_dir=OUTPUT_ROOT / "dist" / "run1",
    visualizers=[
        MixingDiagramVisualizer(),
        MixingDiagram3DVisualizer(),
        LogLikelihoodVisualizer(),
        HistogramVisualizer(),
    ],
)


runs disponiveis: ['run1', 'run2', 'run3', 'run4', 'run5', 'run6', 'run7', 'run8', 'run9']
pipeline: ['Centering', 'Whitening']
convergiu: True  |  247 iteracoes  |  0.080s
curtose das misturas   (antes):  [ 0.956 -1.078 -0.585]
curtose das fontes     (depois): [ 0.203  1.857 -1.184]
log-verossimilhanca final:       -2.729


## 3. Amostra de imagens

Mesmo caso: pixels serializados nao tem ordem temporal significativa
(Secao 13.1.1) e o numero de misturas ja e igual ao de fontes -- so os
passos normais.

In [5]:
imagens_root = DATA_ROOT / "imagens"
print("runs disponiveis:", ImageTemplate.discover_runs(imagens_root))

imagens_data = ImageTemplate(run="run1", data_root=imagens_root)
imagens_model, imagens_metrics = fit_and_report(
    imagens_data,
    build_pipeline(),
    output_dir=OUTPUT_ROOT / "imagens" / "run1",
    visualizers=[
        MixingDiagramVisualizer(),
        MixingDiagram3DVisualizer(),
        LogLikelihoodVisualizer(),
        ImageVisualizer(data=imagens_data),
    ],
)


runs disponiveis: ['run1', 'run3']


pipeline: ['Centering', 'Whitening']
convergiu: True  |  480 iteracoes  |  1.943s
curtose das misturas   (antes):  [ 0.544 -0.281  0.607]
curtose das fontes     (depois): [ 0.983  2.008 -0.586]
log-verossimilhanca final:       -2.648


## 4. Amostra de audio -- com e sem filtragem temporal

Esta e a amostra onde a filtragem temporal (Secao 13.1) de fato se aplica:
os microfones gravam um sinal com ordem temporal real. Comparamos o
pipeline normal contra um pipeline com `TemporalFiltering(kind="high")`
(passa-alta / aproximacao do processo de inovacao) adicionado antes do
branqueamento -- lembrando que, por ser `estimation_only`, o filtro so
influencia a *estimacao* da matriz de separacao B; as fontes finais
(`model.sources_`) sao sempre reconstruidas a partir do audio original,
no comprimento e amplitude originais (Secao 13.1, p.264).

In [6]:
audio_root = DATA_ROOT / "audio"
print("runs disponiveis:", AudioTemplate.discover_runs(audio_root))

print("--- pipeline normal (sem filtragem) ---")
audio_baseline_data = AudioTemplate(run="run1", data_root=audio_root)
baseline_model, baseline_metrics = fit_and_report(
    audio_baseline_data,
    build_pipeline(),
    output_dir=OUTPUT_ROOT / "audio" / "run1_normal",
)

print("\n--- pipeline com passa-alta (TemporalFiltering) ---")
audio_filtered_data = AudioTemplate(run="run1", data_root=audio_root)
filtered_model, filtered_metrics = fit_and_report(
    audio_filtered_data,
    build_pipeline(temporal_filtering=TemporalFiltering(kind="high")),
    output_dir=OUTPUT_ROOT / "audio" / "run1_highpass",
)

print("\nmesmo comprimento do audio original em ambos os casos:")
print(" normal:     ", baseline_model.sources_.shape, "vs mixtures", baseline_model.mixtures_.shape)
print(" passa-alta: ", filtered_model.sources_.shape, "vs mixtures", filtered_model.mixtures_.shape)


runs disponiveis: ['run1', 'run2', 'run3', 'run4', 'run5', 'run6', 'run7', 'run8']
--- pipeline normal (sem filtragem) ---


pipeline: ['Centering', 'Whitening']
convergiu: True  |  52 iteracoes  |  8.017s
curtose das misturas   (antes):  [2.837 3.528]
curtose das fontes     (depois): [3.254 4.938]
log-verossimilhanca final:       -2.658

--- pipeline com passa-alta (TemporalFiltering) ---


pipeline: ['Centering', 'TemporalFiltering', 'Whitening']
convergiu: True  |  23 iteracoes  |  3.694s
curtose das misturas   (antes):  [2.837 3.528]
curtose das fontes     (depois): [3.254 4.939]
log-verossimilhanca final:       -2.237

mesmo comprimento do audio original em ambos os casos:
 normal:      (2, 661500) vs mixtures (2, 661500)
 passa-alta:  (2, 661500) vs mixtures (2, 661500)


A comparacao acima (curtose media das fontes, log-verossimilhanca final,
numero de iteracoes) e a base para decidir, por run de audio, se vale a
pena manter a filtragem temporal no pipeline -- em runs onde ela nao
melhora a nao-gaussianidade nem a convergencia, o passo e simplesmente
omitido (`build_pipeline()` sem `temporal_filtering`), sem qualquer
mudanca no restante do codigo. Escolhemos a run com maior curtose media
em modulo (mais nao-gaussiana => melhor separacao esperada) para exportar
os `.wav` finais.

In [7]:
baseline_score = np.mean(np.abs(baseline_metrics["non_gaussianity_kurtosis"]))
filtered_score = np.mean(np.abs(filtered_metrics["non_gaussianity_kurtosis"]))
print(f"curtose media |k| -- normal: {baseline_score:.3f}  |  passa-alta: {filtered_score:.3f}")

if filtered_score > baseline_score:
    print("=> mantendo TemporalFiltering(kind='high') para esta run; exportando .wav")
    winning_model, winning_data = filtered_model, audio_filtered_data
else:
    print("=> pipeline normal ja e suficiente para esta run; exportando .wav")
    winning_model, winning_data = baseline_model, audio_baseline_data

AudioVisualizer(data=winning_data).plot(winning_model, OUTPUT_ROOT / "audio" / "run1_final")


curtose media |k| -- normal: 4.096  |  passa-alta: 4.096
=> mantendo TemporalFiltering(kind='high') para esta run; exportando .wav


[PosixPath('../output/notebook_demo/audio/run1_final/audio_formas_de_onda_e_espectrogramas.png'),
 PosixPath('../output/notebook_demo/audio/run1_final/fonte_recuperada_1.wav'),
 PosixPath('../output/notebook_demo/audio/run1_final/fonte_recuperada_2.wav')]

## Resumo: pipeline usado por amostra

| Amostra      | Run  | Passos                                             | Motivo |
|--------------|------|-----------------------------------------------------|--------|
| Distribuicoes| run1 | Centering, Whitening                                 | sem estrutura temporal; numero de misturas = numero de fontes |
| Imagens      | run1 | Centering, Whitening                                 | pixels serializados sem ordem temporal; sem excesso de sensores |
| Audio        | run1 | Centering, [TemporalFiltering opcional], Whitening    | estrutura temporal real -- filtragem testada e mantida so se melhorar a curtose/convergencia |
| (demonstracao)| --  | Centering, PCA(n_components=2), Whitening             | exemplo sintetico com mais sensores (5) do que fontes (2) |

`build_pipeline(temporal_filtering=None, pca=None)` deixa explicito, por
run, quais passos opcionais do livro-texto (Secao 13) estao ativos --
bastando adicionar ou remover o argumento correspondente para adaptar o
pipeline a uma nova run, sem tocar em `ICAModel` nem em nenhuma outra
classe do pacote.